# Längenregression

Tabellen 18/19 und Auto-CoT–Plain bei 75 Wörtern. Binomiales GLM mit Logit-Link, Methoden×Stufen-Interzepten, Artikel-Fixed-Effects und einem linearen Wortlängenkoeffizienten je Stufe. Lead-k ist außerhalb der generativen Regression. Keine Spline-Analyse.

In [1]:
from pathlib import Path
from fractions import Fraction
from collections import Counter
from itertools import combinations
import hashlib, importlib.metadata, json, math, platform, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from threadpoolctl import threadpool_limits
from faktenretention.io import PROJECT_ROOT
from faktenretention.data import load_inputs, panel, length_values, reconstruct_qa, aggregate_articles, CELL, KEY
from faktenretention.protocol import METHODS, STAGES, WINDOWS, CONTRASTS, LABELS
threadpool_limits(limits=1)
SEED, B, N_FACTS = (32, 10000, 20)
DATA_DIR = PROJECT_ROOT / 'data'
OUT = PROJECT_ROOT / 'results/06_laengenregression'
FIG = OUT
OUT.mkdir(parents=True, exist_ok=True)
COLORS = dict(zip(METHODS, ['#777777', '#0072B2', '#E69F00', '#009E73', '#CC79A7', '#D55E00']))
MARKERS = dict(zip(METHODS, ['o', 's', '^', 'D', 'v', 'P']))
LINESTYLES = dict(zip(METHODS, ['--', '-', '-.', '--', ':', '-']))
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_rows', 26)

def export(frame, name):
    frame.to_csv(OUT / f'{name}.csv', index=False)

def fmt(x, d=2):
    return f'{x:.{d}f}'.replace('.', ',')

def savefig(fig, name):
    fig.savefig(FIG / f'{name}.png', dpi=220, bbox_inches='tight')
    plt.show()
    plt.close(fig)

In [2]:
inputs = load_inputs()
qa, article = panel(inputs)
facts = inputs['facts']
summaries = inputs['summaries']
articles = inputs['articles']
IDS = sorted(articles.article_id)
assert len(IDS) == 100

In [3]:
import time, warnings
import scipy.linalg as la
from scipy.special import expit
import statsmodels.api as sm
GENERATIVE = METHODS[1:]
MIN_SUPPORT = 20
NAMES = LABELS
software = {'python': platform.python_version(), **{m: importlib.metadata.version(m) for m in ['numpy',
    'pandas', 'scipy', 'statsmodels', 'patsy', 'matplotlib', 'nbformat', 'nbclient', 'nbconvert', 'ipykernel']}}
from faktenretention.statistics import contrasts_for
original_contrasts = contrasts_for(article)

## Überlappung und lokale Unterstützung

Gemeinsamer Bereich = Schnitt der 5.–95. Perzentile. Erforderlich: positive Breite, mindestens 20 Beobachtungen je Verfahren im Bereich und 20 gepaarte Artikel. Jede beobachtete Referenzlänge jedes gepaarten Artikels muss bei **jedem** Verfahren mindestens fünf Ausgaben im Band `max(5 Wörtern, 10 % der Referenzlänge)` haben. Gemeinsame Standardisierung über beide beobachteten Längen derselben gestützten Artikel.

In [4]:
quantiles = article.groupby(['method', 'stage_target_words']).W.quantile([0, 0.05, 0.1, 0.25, 0.5, 0.75,
    0.9, 0.95, 1]).unstack()
quantiles.columns = ['min', 'p05', 'p10', 'p25', 'p50', 'p75', 'p90', 'p95', 'max']
quantiles = quantiles.reset_index()
export(quantiles, 'B_wortzahlverteilungen_24_gruppen')
display(quantiles.round(2))
SUPPORT_DEFS = {'central95': (0.05, 0.95)}
REPORTED={(frozenset([f,s]),t) for f,s in CONTRASTS for t in STAGES} | {(frozenset(['auto_cot','plain']),75)}

def support_for(methods, stage, definition):
    sub = article[article.stage_target_words.eq(stage) & article.method.isin(methods)]
    wide = sub.pivot(index='article_id', columns='method', values='W').reindex(index=IDS, columns=methods)
    loq, hiq = SUPPORT_DEFS[definition]
    lower = float(max((wide[m].quantile(loq) for m in methods)))
    upper = float(min((wide[m].quantile(hiq) for m in methods)))
    in_range = wide.ge(lower) & wide.le(upper) & wide.gt(0)
    n_by_method = {m: int(in_range[m].sum()) for m in methods}
    paired = in_range.all(axis=1)
    local = paired.copy()
    for a in wide.index[paired]:
        for w in wide.loc[a].to_numpy():
            bandwidth = max(5, 0.1 * w)
            if any((int((wide[m] - w).abs().le(bandwidth).sum()) < 5 for m in methods)):
                local.loc[a] = False
                break
    reasons = []
    if lower >= upper:
        reasons.append('kein gemeinsamer Bereich mit positiver Breite')
    if min(n_by_method.values()) < MIN_SUPPORT:
        reasons.append('weniger als 20 Beobachtungen je Verfahren im Bereich')
    if int(local.sum()) < MIN_SUPPORT:
        reasons.append('weniger als 20 lokal gestützte gepaarte Artikel')
    if any((m == 'lead_k' for m in methods)):
        reasons.append('Lead-k bleibt außerhalb der generativen Regression')
    return dict(lower=lower, upper=upper, n_by_method=n_by_method, n_paired_range=int(paired.sum()), n_supported=int(local.sum()),
        ids=wide.index[local].tolist(), estimable=not reasons, reason='; '.join(reasons) if reasons else 'gestützter explorativer Vergleich',
        small_group=0 < int(local.sum()) < 30)
support_rows = []
supports = {}
for first, second in combinations(GENERATIVE, 2):
    for stage in STAGES:
        if (frozenset([first,second]),stage) not in REPORTED:continue
        left = article[(article.method == first) & article.stage_target_words.eq(stage)].set_index('article_id')
        right = article[(article.method == second) & article.stage_target_words.eq(stage)].set_index('article_id')
        direct = (left.W + right.W).gt(0) & (40 * (left.W - right.W).abs()).le(left.W + right.W)
        for definition in SUPPORT_DEFS:
            support = support_for([first, second], stage, definition)
            supports[first, second, stage, definition] = support
            support_rows.append(dict(first=first, second=second, stage_target_words=stage, definition=definition,
                common_lower=support['lower'], common_upper=support['upper'], n_first_in_range=support['n_by_method'][first],
                n_second_in_range=support['n_by_method'][second], n_paired_range=support['n_paired_range'],
                n_supported_articles=support['n_supported'], n_direct_5pct_pairs=int(direct.sum()), estimable=support['estimable'],
                small_group=support['small_group'], status=support['reason']))
overlap = pd.DataFrame(support_rows)
export(overlap, 'ueberlappung_generativer_vergleiche')

def get_support(first, second, stage, definition='central95'):
    key = (first, second, stage, definition)
    if key not in supports:
        key = (second, first, stage, definition)
    return supports[key]
core_support = pd.DataFrame([dict(first=f, second=s, stage_target_words=t, lower=get_support(f, s, t)['lower'],
    upper=get_support(f, s, t)['upper'], n=get_support(f, s, t)['n_supported'], estimable=get_support(f,
    s, t)['estimable'], status=get_support(f, s, t)['reason']) for f, s in CONTRASTS for t in STAGES])
display(core_support.round(2))
export(core_support, 'B_ueberlappung_16_hauptkontraste')
global_support = {t: support_for(GENERATIVE, t, 'central95') for t in STAGES}
display(pd.DataFrame([dict(stage=t, lower=s['lower'], upper=s['upper'], n=s['n_supported'], estimable=s['estimable'],
    status=s['reason']) for t, s in global_support.items()]))

,method,stage_target_words,min,p05,p10,p25,p50,p75,p90,p95,max
0,auto_cot,75,81.0,88.95,95.0,102.75,111.5,123.25,138.5,158.30,185.0
1,auto_cot,150,140.0,160.00,166.0,181.75,209.0,235.50,280.5,304.05,332.0
2,auto_cot,300,206.0,227.55,239.0,272.00,302.5,332.50,360.5,383.00,433.0
3,auto_cot,500,214.0,264.90,282.9,310.75,348.5,383.25,421.6,441.10,491.0
4,automate_cot,75,76.0,87.00,93.9,102.00,110.0,121.50,139.5,162.10,216.0
5,automate_cot,150,140.0,149.95,158.0,174.00,192.5,216.00,243.3,275.30,354.0
6,automate_cot,300,222.0,243.70,258.9,278.00,310.0,346.25,367.9,400.30,435.0
7,automate_cot,500,261.0,327.85,339.7,359.00,390.5,423.75,446.3,464.90,496.0
8,few_shot_cot,75,67.0,78.00,84.7,94.75,102.0,113.25,125.4,131.10,189.0
9,few_shot_cot,150,129.0,140.90,149.0,163.50,180.0,200.00,227.1,246.05,285.0


,first,second,stage_target_words,lower,upper,n,estimable,status
0,zero_shot_cot,plain,500,274.80,279.75,0,False,weniger als 20 Beobachtungen je Verfahren im B...
1,zero_shot_cot,plain,300,218.90,239.85,0,False,weniger als 20 Beobachtungen je Verfahren im B...
2,zero_shot_cot,plain,150,148.85,222.30,33,True,gestützter explorativer Vergleich
3,zero_shot_cot,plain,75,63.00,105.00,38,True,gestützter explorativer Vergleich
4,few_shot_cot,plain,500,236.65,279.75,2,False,weniger als 20 Beobachtungen je Verfahren im B...
5,few_shot_cot,plain,300,217.65,239.85,1,False,weniger als 20 Beobachtungen je Verfahren im B...
6,few_shot_cot,plain,150,140.90,222.30,47,True,gestützter explorativer Vergleich
7,few_shot_cot,plain,75,78.00,131.10,63,True,gestützter explorativer Vergleich
8,auto_cot,few_shot_cot,500,264.90,370.05,48,True,gestützter explorativer Vergleich
9,auto_cot,few_shot_cot,300,227.55,330.10,58,True,gestützter explorativer Vergleich


,stage,lower,upper,n,estimable,status
0,500,327.85,279.75,0,False,kein gemeinsamer Bereich mit positiver Breite;...
1,300,243.70,239.85,0,False,kein gemeinsamer Bereich mit positiver Breite;...
2,150,160.00,222.30,6,False,weniger als 20 lokal gestützte gepaarte Artikel
3,75,88.95,105.00,0,False,weniger als 20 lokal gestützte gepaarte Artikel


In [5]:
global_table = pd.DataFrame([dict(stage_target_words=t, n_supported=s['n_supported'], estimable=s['estimable'],
    lower=s['lower'], upper=s['upper'], reason=s['reason']) for t, s in global_support.items()])
assert not global_table.estimable.any()
export(global_table, 'keine_gemeinsame_fuenfmethoden_rangfolge')

## Modellanpassung und numerische Diagnostik

In [6]:
model_data = article[article.method.isin(GENERATIVE)].sort_values(CELL).reset_index(drop=True).copy()
assert len(model_data) == 2000
MODEL_CELLS = [(m, t) for m in GENERATIVE for t in STAGES]
cell_map = {key: i for i, key in enumerate(MODEL_CELLS)}
article_map = {a: i for i, a in enumerate(IDS)}
N_ART = len(IDS)
ART_INDEX = model_data.article_id.map(article_map).to_numpy()
Y = np.column_stack([model_data.n_retained, model_data.n_valid - model_data.n_retained]).astype(float)

def design(frame, kind):
    n = len(frame)
    base = np.zeros((n, 20 + 99), dtype=float)
    cell_idx = np.array([cell_map[m, int(t)] for m, t in zip(frame.method, frame.stage_target_words)])
    ai = frame.article_id.map(article_map).to_numpy()
    assert np.isfinite(ai).all()
    ai = ai.astype(int)
    base[np.arange(n), cell_idx] = 1
    has_dummy = ai > 0
    base[np.flatnonzero(has_dummy), 20 + ai[has_dummy] - 1] = 1
    length_columns = []
    for stage in STAGES:
        selected = frame.stage_target_words.eq(stage).to_numpy()
        block = np.zeros((n, 1))
        block[selected, 0] = (frame.loc[selected, 'W'].to_numpy() - stage) / 100
        length_columns.append(block)
    return np.column_stack([base] + length_columns)
models = {}
diagnostic_rows = []
coefficient_rows = []
for kind in ['linear']:
    X = design(model_data, kind)
    names = [f'{m}:{t}' for m, t in MODEL_CELLS] + [f'article_{a}' for a in IDS[1:]]
    names += [f'length_{t}' for t in STAGES]
    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        result = sm.GLM(Y, X, family=sm.families.Binomial()).fit(maxiter=100, tol=1e-10, cov_type='cluster',
            cov_kwds={'groups': model_data.article_id.to_numpy(), 'use_correction': True})
    seconds = time.perf_counter() - started
    cov = np.asarray(result.cov_params())
    eig = la.eigvalsh((cov + cov.T) / 2)
    stable = bool(result.converged and np.isfinite(result.params).all() and (np.linalg.matrix_rank(X) == X.shape[1]) and np.isfinite(cov).all() and (eig.min() > -1e-08))
    models[kind] = dict(X=X, result=result, beta=np.asarray(result.params), cov=cov, stable=stable, names=names)
    diagnostic_rows.append(dict(model=kind, n_rows=len(X), n_parameters=X.shape[1], design_rank=np.linalg.matrix_rank(X),
        n_article_clusters=N_ART, converged=result.converged, iterations=result.fit_history.get('iteration'),
        max_abs_coefficient=float(abs(result.params).max()), design_condition=float(np.linalg.cond(X)),
        covariance_rank=np.linalg.matrix_rank(cov), min_covariance_eigenvalue=float(eig.min()), max_abs_score=float(abs(result.model.score(result.params)).max()),
        pearson_ratio=float(result.pearson_chi2 / result.df_resid), deviance=float(result.deviance),
        fit_seconds=seconds, warning_text=' | '.join((str(w.message) for w in captured)), stable=stable))
    for i, name in enumerate(names):
        coefficient_rows.append(dict(model=kind, term=name, estimate=result.params[i], cluster_robust_se=float(np.sqrt(max(cov[i, i], 0)))))
    influence = result.get_influence(observed=False)
    diagnostics = model_data[CELL + ['W', 'n_retained', 'n_valid', 'retention_observed']].copy()
    diagnostics['predicted'] = result.predict()
    diagnostics['pearson_residual'] = result.resid_pearson
    diagnostics['leverage'] = influence.hat_matrix_diag
    export(diagnostics, f'C_residuen_und_hebelwerte_{kind}')
diagnostics_table = pd.DataFrame(diagnostic_rows)
export(diagnostics_table, 'C_modelldiagnostik')
export(pd.DataFrame(coefficient_rows), 'C_regressionskoeffizienten_clusterrobuste_se')
display(diagnostics_table)
assert diagnostics_table.stable.all(), 'Modellinstabilität: keine verlässlichen Standardisierungen ausgeben.'

,model,n_rows,n_parameters,design_rank,n_article_clusters,converged,iterations,max_abs_coefficient,design_condition,covariance_rank,min_covariance_eigenvalue,max_abs_score,pearson_ratio,deviance,fit_seconds,warning_text,stable
0,linear,2000,123,123,100,True,5,1.997302,111.212716,23,-1.119117e-15,1.631556e-11,0.65107,1270.384288,0.047553,,True


## Gemeinsame Standardisierung

In [7]:
comparison_specs = []
for first, second in combinations(GENERATIVE, 2):
    for stage in STAGES:
        if (frozenset([first,second]),stage) not in REPORTED:continue
        for definition in SUPPORT_DEFS:
            support = get_support(first, second, stage, definition)
            if not support['estimable']:
                continue
            ids = support['ids']
            left = article[(article.method == first) & article.stage_target_words.eq(stage)].set_index('article_id').loc[ids]
            right = article[(article.method == second) & article.stage_target_words.eq(stage)].set_index('article_id').loc[ids]
            reference = pd.DataFrame({'article_id': np.repeat(ids, 2), 'W': np.column_stack([left.W, right.W]).ravel(),
                'stage_target_words': stage})
            comparison_specs.append(dict(first=first, second=second, stage=stage, definition=definition,
                support=support, reference=reference, reference_article_indices=reference.article_id.map(article_map).to_numpy(),
                raw_same_articles_pp=float(100 * (left.retention_observed - right.retention_observed).mean())))

def standardized_point(spec, kind):
    ref = spec['reference']
    xa = design(ref.assign(method=spec['first']), kind)
    xb = design(ref.assign(method=spec['second']), kind)
    beta = models[kind]['beta']
    return (float(100 * np.mean(expit(xa @ beta) - expit(xb @ beta))), xa, xb)
point_rows = []
for spec in comparison_specs:
    for kind in models:
        point, xa, xb = standardized_point(spec, kind)
        point_rows.append(dict(first=spec['first'], second=spec['second'], stage_target_words=spec['stage'],
            definition=spec['definition'], model=kind, adjusted_difference_pp=point, raw_same_articles_pp=spec['raw_same_articles_pp'],
            n_articles=spec['support']['n_supported'], common_lower=spec['support']['lower'], common_upper=spec['support']['upper']))
points = pd.DataFrame(point_rows)
display(points[(points.definition == 'central95') & (points.model == 'linear')].round(3))

,first,second,stage_target_words,definition,model,adjusted_difference_pp,raw_same_articles_pp,n_articles,common_lower,common_upper
0,plain,zero_shot_cot,150,central95,linear,-2.799,-5.262,33,148.85,222.30
1,plain,zero_shot_cot,75,central95,linear,-0.779,1.171,38,63.00,105.00
2,plain,few_shot_cot,150,central95,linear,-0.849,-4.269,47,140.90,222.30
3,plain,few_shot_cot,75,central95,linear,-1.255,-1.416,63,78.00,131.10
4,plain,auto_cot,75,central95,linear,-3.187,-3.256,59,88.95,156.10
5,few_shot_cot,auto_cot,500,central95,linear,-2.821,-4.797,48,264.90,370.05
6,few_shot_cot,auto_cot,300,central95,linear,-3.369,-7.067,58,227.55,330.10
7,few_shot_cot,auto_cot,150,central95,linear,-2.419,-3.970,52,160.00,246.05
8,few_shot_cot,auto_cot,75,central95,linear,-1.882,-3.450,64,88.95,131.10
9,auto_cot,automate_cot,500,central95,linear,1.912,0.472,44,327.85,441.10


## Artikelbootstrap und Cache

Der Cache enthält 10.000 vollständige Anpassungen. Der Fingerprint prüft Designmatrix, binomiale Zähler, Modellparameter, Ziehungsgewichte, numerischen Worker und Paketversionen. Standardmäßig wird nur der passende Cache gelesen. `RECOMPUTE_BOOTSTRAP=True` führt ausdrücklich alle 10.000 Anpassungen erneut aus und schreibt einen neuen Cache nach `results/`; die Quelle in `data/` bleibt unverändert.

In [8]:
RECOMPUTE_BOOTSTRAP = False

In [9]:
from faktenretention import analysis_bootstrap_worker as worker
worker_source = Path(worker.__file__).read_text()
from faktenretention import analysis_bootstrap_worker as worker
draws = np.random.default_rng(SEED).integers(0, N_ART, size=(B, N_ART))
bootstrap_counts = np.zeros((B, N_ART), dtype=np.int16)
np.add.at(bootstrap_counts, (np.repeat(np.arange(B), N_ART), draws.ravel()), 1)
assert bootstrap_counts.sum(axis=1).tolist() == [N_ART] * B
boot_coefficients = {}
bootstrap_fit_rows = []
weight_validation = []
for kind, state in models.items():
    worker.initialize(state['X'], Y, ART_INDEX, state['beta'])
    test_indices = [int(np.flatnonzero(bootstrap_counts[:, 0] > 0)[0]), int(np.flatnonzero(bootstrap_counts[:, 0] == 0)[0])]
    for draw in test_indices:
        counts = bootstrap_counts[draw]
        reference = int(np.flatnonzero(counts > 0)[0])
        columns = np.ones(state['X'].shape[1], dtype=bool)
        for ai in range(1, N_ART):
            if counts[ai] == 0 or ai == reference:
                columns[20 + ai - 1] = False
        expanded = np.repeat(np.arange(len(model_data)), counts[ART_INDEX])
        direct_fit = sm.GLM(Y[expanded], state['X'][expanded][:, columns], family=sm.families.Binomial()).fit(maxiter=100, tol=1e-10)
        weighted, records_check = worker.fit_batch(counts[None, :])
        included = counts[ART_INDEX] > 0
        direct_predictions = direct_fit.predict(state['X'][included][:, columns])
        weighted_predictions = expit(state['X'][included] @ weighted[0])
        max_error = float(abs(direct_predictions - weighted_predictions).max())
        assert records_check[0][0] and direct_fit.converged and (max_error < 1e-07)
        weight_validation.append(dict(model=kind, draw=draw, original_reference_absent=bool(counts[0] == 0),
            n_expanded_rows=len(expanded), max_prediction_difference=max_error, passed=True))
export(pd.DataFrame(weight_validation), 'bootstrap_haufigkeitsgewichte_unabhaengig_validiert')
for kind, state in models.items():
    fingerprint = hashlib.sha256(state['X'].tobytes() + Y.tobytes() + state['beta'].tobytes() + bootstrap_counts.tobytes() + worker_source.encode() + json.dumps(software,
        sort_keys=True).encode()).hexdigest()
    cache = DATA_DIR / 'bootstrap_linear.npz'
    cached = False
    if cache.is_file() and (not RECOMPUTE_BOOTSTRAP):
        with np.load(cache, allow_pickle=False) as archive:
            if str(archive['fingerprint']) == fingerprint:
                coefficients = archive['coefficients'].copy()
                records = json.loads(str(archive['records']))
                cached = True
    started = time.perf_counter()
    if not cached:
        if not RECOMPUTE_BOOTSTRAP:
            raise RuntimeError('Cache passt nicht: dokumentierte Python-/Paketversionen verwenden oder RECOMPUTE_BOOTSTRAP ausdrücklich aktivieren.')
        worker.initialize(state['X'], Y, ART_INDEX, state['beta'])
        batches = [worker.fit_batch(bootstrap_counts[start:start + 100]) for start in range(0, B, 100)]
        coefficients = np.concatenate([batch[0] for batch in batches])
        records = [r for batch in batches for r in batch[1]]
        np.savez_compressed(OUT / 'bootstrap_linear_neu.npz', coefficients=coefficients, records=json.dumps(records),
            fingerprint=fingerprint, bootstrap_counts=bootstrap_counts)
    assert coefficients.shape == (B, state['X'].shape[1]) and len(records) == B
    success = np.array([bool(r[0]) for r in records])
    assert np.isfinite(coefficients[success]).all()
    boot_coefficients[kind] = coefficients
    bootstrap_fit_rows.append(dict(model=kind, n_requested=B, n_success=int(success.sum()), n_failed=int((~success).sum()),
        n_warning_fits=sum((bool(r[2]) for r in records)), max_iterations=max((r[1] for r in records)),
        seconds_this_run=time.perf_counter() - started, cache_used=cached, fingerprint=fingerprint))
    failures = pd.DataFrame([dict(draw=i, converged=r[0], iterations=r[1], message=r[2]) for i, r in enumerate(records) if not r[0] or r[2]],
        columns=['draw', 'converged', 'iterations', 'message'])
    export(failures, f'bootstrap_numerische_hinweise_{kind}')
bootstrap_fits = pd.DataFrame(bootstrap_fit_rows)
export(bootstrap_fits, 'bootstrap_anpassungsdiagnostik')
display(bootstrap_fits.drop(columns='fingerprint'))

,model,n_requested,n_success,n_failed,n_warning_fits,max_iterations,seconds_this_run,cache_used
0,linear,10000,10000,0,0,4,0.001324,True


## Tabelle 18: Hauptkontraste

In [10]:
bootstrap_comparisons = []
for spec in comparison_specs:
    for kind in models:
        point, xa, xb = standardized_point(spec, kind)
        reference_ai = spec['reference_article_indices']
        distributions = np.full(B, np.nan)
        for start in range(0, B, 200):
            end = min(start + 200, B)
            coefficients = boot_coefficients[kind][start:end]
            weights = bootstrap_counts[start:end, reference_ai]
            denominator = weights.sum(axis=1)
            probabilities_a = expit(coefficients @ xa.T)
            probabilities_b = expit(coefficients @ xb.T)
            numerator = ((probabilities_a - probabilities_b) * weights).sum(axis=1)
            distributions[start:end] = np.divide(100 * numerator, denominator, out=np.full(end - start,
                np.nan), where=denominator > 0)
        finite = np.isfinite(distributions)
        reliable = int(finite.sum()) >= int(0.99 * B)
        low, high = np.nanquantile(distributions, [0.025, 0.975]) if reliable else [np.nan, np.nan]
        bootstrap_comparisons.append(dict(first=spec['first'], second=spec['second'], stage_target_words=spec['stage'],
            definition=spec['definition'], model=kind, adjusted_difference_pp=point if reliable else np.nan,
            ci_low_pp=low, ci_high_pp=high, n_bootstrap_valid=int(finite.sum()), n_bootstrap_missing=int((~finite).sum()),
            reliable=reliable, n_articles=spec['support']['n_supported'], n_reference_points=len(spec['reference']),
            common_lower=spec['support']['lower'], common_upper=spec['support']['upper'], raw_same_articles_pp=spec['raw_same_articles_pp']))
controlled = pd.DataFrame(bootstrap_comparisons)
export(controlled, 'D_alle_gestuetzten_kontraste_95ci')

def oriented_result(first, second, stage, definition='central95', kind='linear'):
    rows = controlled[(controlled['first'] == first) & (controlled.second == second) & controlled.stage_target_words.eq(stage) & controlled.definition.eq(definition) & controlled.model.eq(kind)]
    if len(rows):
        return rows.iloc[0].copy()
    rows = controlled[(controlled['first'] == second) & (controlled.second == first) & controlled.stage_target_words.eq(stage) & controlled.definition.eq(definition) & controlled.model.eq(kind)]
    if not len(rows):
        return None
    row = rows.iloc[0].copy()
    row['first'], row['second'] = (first, second)
    row['adjusted_difference_pp'] = -row.adjusted_difference_pp
    row['raw_same_articles_pp'] = -row.raw_same_articles_pp
    row['ci_low_pp'], row['ci_high_pp'] = (-row.ci_high_pp, -row.ci_low_pp)
    return row
core_rows = []
for original in original_contrasts.itertuples():
    first, second, stage = (original.first, original.second, original.stage_target_words)
    support = get_support(first, second, stage)
    adjusted = oriented_result(first, second, stage)
    reliable = adjusted is not None and bool(adjusted.reliable)
    core_rows.append(dict(first=first, second=second, contrast=original.contrast, stage_target_words=stage,
        original_difference_pp=original.mean, original_ci_low_pp=original.ci_low, original_ci_high_pp=original.ci_high,
        controlled_difference_pp=adjusted.adjusted_difference_pp if reliable else np.nan, controlled_ci_low_pp=adjusted.ci_low_pp if reliable else np.nan,
        controlled_ci_high_pp=adjusted.ci_high_pp if reliable else np.nan, raw_same_articles_pp=adjusted.raw_same_articles_pp if reliable else np.nan,
        common_lower=support['lower'], common_upper=support['upper'], n_first_in_range=support['n_by_method'][first],
        n_second_in_range=support['n_by_method'][second], n_supported_articles=support['n_supported'],
        n_reference_points=2 * support['n_supported'], reliable=reliable, status=('kleine Gruppe (<30 Artikel)' if support['small_group'] else 'gestützt') if reliable else 'nicht zuverlässig schätzbar: ' + support['reason']))
core_table = pd.DataFrame(core_rows)
export(core_table, 'haupttabelle_16_originale_und_laengenkontrollierte_kontraste')
display(core_table.drop(columns=['first', 'second']).round(3))

,contrast,stage_target_words,original_difference_pp,original_ci_low_pp,original_ci_high_pp,controlled_difference_pp,controlled_ci_low_pp,controlled_ci_high_pp,raw_same_articles_pp,common_lower,common_upper,n_first_in_range,n_second_in_range,n_supported_articles,n_reference_points,reliable,status
0,Zero-Shot-CoT − Plain,500,16.997,14.294,19.768,NaN,NaN,NaN,NaN,274.80,279.75,4,3,0,0,False,nicht zuverlässig schätzbar: weniger als 20 Be...
1,Zero-Shot-CoT − Plain,300,17.843,15.113,20.575,NaN,NaN,NaN,NaN,218.90,239.85,6,11,0,0,False,nicht zuverlässig schätzbar: weniger als 20 Be...
2,Zero-Shot-CoT − Plain,150,8.523,5.587,11.325,2.799,0.106,5.513,5.262,148.85,222.30,77,47,33,66,True,gestützt
3,Zero-Shot-CoT − Plain,75,-4.874,-7.574,-2.247,0.779,-1.381,3.017,-1.171,63.00,105.00,89,49,38,76,True,gestützt
4,Few-Shot-CoT − Plain,500,9.299,6.754,11.883,NaN,NaN,NaN,NaN,236.65,279.75,20,17,2,4,False,nicht zuverlässig schätzbar: weniger als 20 Be...
5,Few-Shot-CoT − Plain,300,11.032,8.513,13.572,NaN,NaN,NaN,NaN,217.65,239.85,14,11,1,2,False,nicht zuverlässig schätzbar: weniger als 20 Be...
6,Few-Shot-CoT − Plain,150,5.719,3.275,8.147,0.849,-1.411,3.134,4.269,140.90,222.30,82,57,47,94,True,gestützt
7,Few-Shot-CoT − Plain,75,0.750,-1.676,3.170,1.255,-0.840,3.461,1.416,78.00,131.10,91,69,63,126,True,gestützt
8,Auto-CoT − Few-Shot-CoT,500,7.146,4.894,9.481,2.821,0.568,5.085,4.797,264.90,370.05,60,83,48,96,True,gestützt
9,Auto-CoT − Few-Shot-CoT,300,7.119,4.980,9.318,3.369,1.377,5.444,7.067,227.55,330.10,69,84,58,116,True,gestützt


## Auto-CoT–Plain bei 75 Wörtern und Spezifikation

In [11]:
extra = oriented_result('auto_cot', 'plain', 75)
assert extra is not None and extra.reliable
export(pd.DataFrame([extra]), 'auto_cot_plain_75')
display(pd.DataFrame([extra]).round(4))
specification = pd.DataFrame([('Modell', 'Gruppiertes binomiales GLM, Logit-Link'), ('Effekte', 'Methoden×Stufen-Zellinterzepte und 99 Artikel-Fixed-Effects'),
    ('Textlänge', 'Ein linearer Koeffizient pro Stufe, (W−Ziel)/100'), ('Bereich', 'Schnitt der 5.–95. Perzentile'),
    ('Unterstützung', '≥20 Beobachtungen je Methode und ≥20 lokal gestützte gepaarte Artikel; ≥5 Ausgaben je Methode im lokalen Band max(5,0.1W)'),
    ('Standardisierung', 'Beide beobachteten Wortlängen je gestütztem Artikel, gleiche Artikelgewichte'),
    ('Bootstrap', '10000 Artikelziehungen, Seed 32, 95-%-Perzentilintervalle')], columns=['Aspekt', 'Definition'])
export(specification, 'tabelle_19_spezifikation')

,first,second,stage_target_words,definition,model,adjusted_difference_pp,ci_low_pp,ci_high_pp,n_bootstrap_valid,n_bootstrap_missing,reliable,n_articles,n_reference_points,common_lower,common_upper,raw_same_articles_pp
4,auto_cot,plain,75,central95,linear,3.1868,1.3558,5.0402,10000,0,True,59,118,88.95,156.1,3.2555
